# Lesson 17: Local Edge RAG Pipelines — On-Device Embeddings, Semantic Retrieval & RAGAS Evaluation

In this companion notebook, you will:
1. Build a privacy-preserving local RAG pipeline using `sentence-transformers/all-MiniLM-L6-v2` embeddings.
2. Ground a compact local SLM (with optional `gemini-2.5-flash` hybrid fallback) in retrieved context.
3. Evaluate retrieval and generation quality using **RAGAS** metrics: Faithfulness, Answer Relevancy, and Context Precision.

In [ ]:
!pip install -q -U sentence-transformers numpy google-genai

## 1. Local Vector Index with `sentence-transformers`
Embed an on-device knowledge base in memory and retrieve top-k passages via cosine similarity.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

docs = [
    "INT4 quantization reduces model weight size by 75% compared to FP16, enabling 3.8B SLMs to fit in 2 GB RAM.",
    "STDIO MCP servers run as isolated local subprocesses with zero open TCP ports on the network.",
    "PagedAttention divides the KV cache into fixed-size blocks allocated on demand, cutting fragmentation under 4%."
]

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vecs = embedder.encode(docs, normalize_embeddings=True)

query = "How does INT4 help small models fit in device RAM?"
q_vec = embedder.encode([query], normalize_embeddings=True)[0]
scores = doc_vecs @ q_vec
top_idx = int(np.argmax(scores))
print(f"Top Match (score={scores[top_idx]:.4f}): {docs[top_idx]}")

## 2. Compute RAGAS Quality Metrics
Evaluate Faithfulness ($|V_{\text{supported}}| / |V_{\text{claims}}|$) and Context Precision@K on the retrieved answer.

In [ ]:
claims = [
    ("INT4 reduces weight size by 75% vs FP16", True),
    ("3.8B SLMs fit in 2 GB RAM", True),
]
faithfulness = sum(1 for _, ok in claims if ok) / len(claims)
print(f"RAGAS Faithfulness: {faithfulness:.2f}")